## **Impor Pustaka (Library Setup)**

In [ ]:
import pandas as pd
import glob
import os
import warnings

warnings.filterwarnings('ignore')

## **Ekstraksi Data (Data Loading)**

In [2]:
raw_path = os.path.join('..', 'data_raw')
all_files = glob.glob(os.path.join(raw_path, "Streaming_History_Audio_*.json"))

print(f"Ditemukan {len(all_files)} file JSON. Sedang menggabungkan...")

# Membaca dan menggabungkan semua file
df_list = [pd.read_json(file) for file in all_files]
df_raw = pd.concat(df_list, ignore_index=True)

print(f"Total baris mentah: {df_raw.shape[0]}")

Ditemukan 16 file JSON. Sedang menggabungkan...
Total baris mentah: 167570


## **Pembersihan & Rekayasa Fitur (Data Cleaning & Feature Engineering)**

In [6]:
# Memilih kolom utama yang relevan
columns_to_keep = [
    'ts', 'ms_played', 'master_metadata_track_name', 
    'master_metadata_album_artist_name', 'master_metadata_album_album_name',
    'reason_start', 'reason_end', 'skipped'
]
df = df_raw[columns_to_keep].copy()

# Mengubah nama kolom agar lebih intuitif
df.rename(columns={
    'ts': 'timestamp',
    'ms_played': 'duration_ms',
    'master_metadata_track_name': 'track_name',
    'master_metadata_album_artist_name': 'artist_name',
    'master_metadata_album_album_name': 'album_name'
}, inplace=True)

# Membersihkan baris tanpa nama lagu (seperti Podcast atau file lokal yang tidak dikenali)
df.dropna(subset=['track_name', 'artist_name'], inplace=True)

# Rekayasa Waktu (Timezone & Komponen Waktu)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'] = df['timestamp'].dt.tz_convert('Asia/Jakarta')

df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day_name'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

# Rekayasa Durasi (Milidetik ke Menit)
df['duration_min'] = df['duration_ms'] / 60000

# Penyaringan Logika: Hanya ambil lagu yang didengarkan lebih dari 30 detik
df_clean = df[df['duration_ms'] >= 30000].copy()

# Menghapus informasi zona waktu sebelum diekspor
df_clean['timestamp'] = df_clean['timestamp'].dt.tz_localize(None)

print(f"Total baris setelah pembersihan: {df_clean.shape[0]}")
display(df_clean.sample(3))

Total baris setelah pembersihan: 144185


,timestamp,duration_ms,track_name,artist_name,album_name,reason_start,reason_end,skipped,year,month,day_name,hour,duration_min
81922,2023-10-10 13:59:14,284304,How,Morad,About a Woman,trackdone,trackdone,False,2023,10,Tuesday,13,4.738400
127845,2024-05-29 17:26:07,311813,Bat Country,Avenged Sevenfold,City of Evil,trackdone,trackdone,False,2024,5,Wednesday,17,5.196883
152537,2025-07-18 21:22:15,263160,Guardians Of Asgaard,Amon Amarth,Twilight Of The Thunder God,trackdone,trackdone,False,2025,7,Friday,21,4.386000


## **Ekspor Data Akhir (Saving to CSV)**

In [7]:
# Menentukan lokasi penyimpanan di folder data_processed
processed_path = os.path.join('..', 'data_processed', 'spotify_clean_ready.csv')

# Mengekspor DataFrame ke file CSV
df_clean.to_csv(processed_path, index=False)

print(f"Data sukses dibersihkan dan disimpan di:\n{processed_path}")

Data sukses dibersihkan dan disimpan di:
..\data_processed\spotify_clean_ready.csv
